# ATML PA2 — Task 4: Safety calibration
Kaggle: GPU T4 x2, Internet on. Everything comes from the GitHub repo plus the pinned course assets.
No task is retrained: the three standard adapters must already be tracked in the repo.

In [ ]:
REPO_URL = "https://github.com/danishrasool10/ATML-PA2.git"
WORK = "/kaggle/working/ATML-PA2"
CFG = "configs/feedback.yaml"
# standard adapters as verified locally; guards against an ablation fork slipping in
EXPECTED_SHA256 = {
    "task2_ppo": "527a925a7d2ae6a877e5884de4cde483fe458f359fb92ba5ca01170f2012f8e6",
    "task3_grpo": "c0c073fe07a6fca66b5c261fd89b7dc64d0b03cdaa49ce2ccbc1d1fd5d36949e",
}
XSTEST_SHA256 = "4f1b18b267a5ad6aeebedd46c5128f61e69a8f367a81626a311ae2cf8c936a20"

!test -d {WORK} || git clone -q {REPO_URL} {WORK}
!git -C {WORK} pull -q --ff-only && git -C {WORK} log --oneline -1
%cd {WORK}
!pip install -q -r requirements.txt
!python -c "import torch, transformers, peft, bitsandbytes; print(torch.__version__, transformers.__version__, peft.__version__, bitsandbytes.__version__, torch.cuda.device_count(), 'GPU(s)')"

In [ ]:
import hashlib
from huggingface_hub import snapshot_download
snapshot_download(repo_id="AbDu11aHHH/ATML-PA2-assets", repo_type="dataset",
                  revision="0b350481fb03f5525a35bcdec4131bd4fe487f98", local_dir=WORK,
                  allow_patterns=["data/**", "manifests/**"])
sha = hashlib.sha256(open(f"{WORK}/data/xstest_safety_prompts.csv", "rb").read()).hexdigest()
assert sha == XSTEST_SHA256, sha
print("course data OK")

In [ ]:
from pathlib import Path

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

missing = [str(p) for t in ("task1_dpo", "task2_ppo", "task3_grpo")
           for p in (Path(WORK) / "outputs" / t / "standard" / f for f in ("adapter_config.json", "adapter_model.safetensors"))
           if not p.exists()]
assert not missing, "Missing from the repo (push them; nothing is retrained here):\n" + "\n".join(missing)
for task in ("task1_dpo", "task2_ppo", "task3_grpo"):
    h = sha256(Path(WORK) / "outputs" / task / "standard" / "adapter_model.safetensors")
    want = EXPECTED_SHA256.get(task)
    assert want is None or h == want, f"{task}: adapter in repo is not the verified standard run"
    print(f"{task:10s} {h[:16]}  {'verified' if want else 'present'}")

## Step 1 — common deterministic generation

In [ ]:
%%bash
set -e
cd /kaggle/working/ATML-PA2
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  CUDA_VISIBLE_DEVICES=0 python -m task4_safety.generate_responses --config configs/feedback.yaml --policies sft dpo > gen_gpu0.log 2>&1 &
  CUDA_VISIBLE_DEVICES=1 python -m task4_safety.generate_responses --config configs/feedback.yaml --policies ppo grpo > gen_gpu1.log 2>&1 &
  wait
  grep -h -E "^\[|Error" gen_gpu0.log gen_gpu1.log || true
fi
# fills any policy a worker failed on; re-checks the shared decoding spec
python -m task4_safety.generate_responses --config configs/feedback.yaml

In [ ]:
import json
m = json.load(open(f"{WORK}/results/task4_safety/generation_manifest.json"))
print(json.dumps(m["common"], indent=1))
for k, v in m["policies"].items():
    print(f"{k:5s} adapter={v['adapter']} sha={str(v['adapter_sha256'])[:12]} tok={v['mean_response_tokens']:.1f} trunc={v['truncation_rate']:.3f}")

## Step 2 — categorical AI judge (resumable: rerun after a disconnect)

In [ ]:
%%bash
set -e
cd /kaggle/working/ATML-PA2
if [ "$(nvidia-smi -L | wc -l)" -ge 2 ]; then
  CUDA_VISIBLE_DEVICES=0 python -m task4_safety.judge_responses --config configs/feedback.yaml --policies sft dpo > judge_gpu0.log 2>&1 &
  CUDA_VISIBLE_DEVICES=1 python -m task4_safety.judge_responses --config configs/feedback.yaml --policies ppo grpo > judge_gpu1.log 2>&1 &
  wait
  grep -h -E "^\[judge:[a-z]+\] \{|Error" judge_gpu0.log judge_gpu1.log || true
fi
python -m task4_safety.judge_responses --config configs/feedback.yaml > judge_final.log 2>&1
grep -h -E "^\[judge:[a-z]+\] \{|Error" judge_final.log || true
python -m task4_safety.evaluate_safety --config configs/feedback.yaml

In [ ]:
from IPython.display import Markdown
Markdown(open(f"{WORK}/results/task4_safety/tables.md").read())

## Step 3 — blind audit worksheet + download

In [ ]:
!python -m task4_safety.make_audit_sheet --config {CFG}
!cp {WORK}/*.log {WORK}/results/task4_safety/ 2>/dev/null; cd {WORK} && zip -qr /kaggle/working/task4_results.zip results/task4_safety
from IPython.display import FileLink, display
display(FileLink("task4_results.zip", result_html_prefix="Results: "))

Locally: unzip `task4_results.zip` into the repo root, fill `results/task4_safety/audit/audit_worksheet.csv` → `Human_Label`
(don't open `audit_key.csv` / `judged_*.jsonl` first), then `python -m task4_safety.analyze_audit --config configs/feedback.yaml`.